In [2]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, Matern, RationalQuadratic, WhiteKernel, ConstantKernel as C
from scipy.stats import norm
from scipy.optimize import minimize

# ----------------------------
# Load data
# ----------------------------
X = np.load(r"C:\Users\nmnko\Desktop\NEW ML 2025\Module 12\Data\initial_data for functions Mini lesson 12.8\function_3\initial_inputs.npy")
y = np.load(r"C:\Users\nmnko\Desktop\NEW ML 2025\Module 12\Data\initial_data for functions Mini lesson 12.8\function_3\initial_outputs.npy")
print(f"Inputs:{X}")
print(f"Outputs:{y}")


Inputs:[[0.17152521 0.34391687 0.2487372 ]
 [0.24211446 0.64407427 0.27243281]
 [0.53490572 0.39850092 0.17338873]
 [0.49258141 0.61159319 0.34017639]
 [0.13462167 0.21991724 0.45820622]
 [0.34552327 0.94135983 0.26936348]
 [0.15183663 0.43999062 0.99088187]
 [0.64550284 0.39714294 0.91977134]
 [0.74691195 0.28419631 0.22629985]
 [0.17047699 0.6970324  0.14916943]
 [0.22054934 0.29782524 0.34355534]
 [0.66601366 0.67198515 0.2462953 ]
 [0.04680895 0.23136024 0.77061759]
 [0.60009728 0.72513573 0.06608864]
 [0.96599485 0.86111969 0.56682913]
 [0.153611   0.478413   1.046184  ]
 [0.153611   0.478413   1.046184  ]
 [0.153762   0.481692   1.050904  ]
 [0.154005   0.486953   1.058471  ]
 [0.15459    0.49954    1.076579  ]
 [0.155115   0.511117   1.093224  ]
 [0.161149   0.508344   1.088945  ]
 [0.161149   0.508344   1.088945  ]]
Outputs:[-0.1121222  -0.08796286 -0.11141465 -0.03483531 -0.04800758 -0.11062091
 -0.39892551 -0.11386851 -0.13146061 -0.09418956 -0.04694741 -0.10596504
 -0.118048

In [3]:

y_best = np.min(y)  # best observed value


def expected_improvement(gp, y_best, x):
    mu, sigma = gp.predict(x.reshape(1,-1), return_std=True)
    s = float(sigma[0])
    if s <= 1e-12:
        return 0.0
    Z = (y_best - mu[0]) / s
    ei = (y_best - mu[0]) * norm.cdf(Z) + s * norm.pdf(Z)
    return float(ei)

def probability_improvement(gp, y_best, x, xi=0.0):
    mu, sigma = gp.predict(x.reshape(1,-1), return_std=True)
    s = float(sigma[0])
    if s <= 1e-12:
        return 0.0
    Z = (y_best - mu[0] - xi) / s
    return float(norm.cdf(Z))

def lower_confidence_bound(gp, x, beta=2.0):
    mu, sigma = gp.predict(x.reshape(1,-1), return_std=True)
    return float(mu[0] - beta * sigma[0])


# Acquisition optimizer

def optimize_acquisition_max(acq_func, gp, bounds, n_restarts=50, dim=3, y_best=y_best):
    rng = np.random.RandomState(42)
    best_x = None
    best_val = -np.inf

    # create starts: random + existing X + corners
    starts = list(rng.rand(n_restarts, dim))
    starts.extend(list(X))
    corners = [[0,0,0],[0,0,1],[0,1,0],[0,1,1],
               [1,0,0],[1,0,1],[1,1,0],[1,1,1]]
    starts.extend(corners)

    for x0 in starts:
        res = minimize(lambda xx: -acq_func(gp, y_best, xx),
                       x0, bounds=bounds, method='L-BFGS-B')
        if not res.success:
            continue
        val = -res.fun
        if val > best_val:
            best_val = val
            best_x = res.x

    return best_x, best_val


# Kernels to test

kernel_list = {
    "RBF": C(1.0, (1e-3, 1e3)) * RBF(length_scale=0.1, length_scale_bounds=(1e-2, 10)) 
           + WhiteKernel(noise_level=1e-3, noise_level_bounds=(1e-3, 0.1)),
    
    "Matern": C(1.0, (1e-3, 1e3)) * Matern(length_scale=0.1, length_scale_bounds=(1e-2, 10), nu=2.5) 
              + WhiteKernel(noise_level=1e-3, noise_level_bounds=(1e-3, 0.1)),
    
    "RationalQuadratic": C(1.0, (1e-3, 1e3)) * RationalQuadratic(length_scale=0.1, alpha=1.0, 
                                                                  length_scale_bounds=(1e-2, 10),
                                                                  alpha_bounds=(0.1, 10)) 
                        + WhiteKernel(noise_level=1e-3, noise_level_bounds=(1e-3, 0.1))
}


# Loop over kernels

bounds = [(0,3),(0,3),(0,3)] 

for name, kernel in kernel_list.items():
    gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=10,
                                  normalize_y=True, random_state=42)
    gp.fit(X, y)
    
    print(f"\nKernel: {name}, Optimized kernel: {gp.kernel_}")
    y_best = np.min(y)

    # Optimize acquisitions
    x_ei, v_ei = optimize_acquisition_max(expected_improvement, gp, bounds, dim=3, y_best=y_best)
    x_pi, v_pi = optimize_acquisition_max(probability_improvement, gp, bounds, dim=3, y_best=y_best)
    x_lcb, v_lcb = optimize_acquisition_max(lambda g,y,x: lower_confidence_bound(g, x, beta=2.5),
                                            gp, bounds, dim=3, y_best=y_best)

    print("Next points and acquisition values:")
    print("EI :", x_ei, "Value:", v_ei)
    print("PI :", x_pi, "Value:", v_pi)
    print("LCB:", x_lcb, "Value:", v_lcb)


C:\Users\nmnko\anaconda3\Lib\site-packages\sklearn\gaussian_process\kernels.py:445: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 0.001. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(



Kernel: RBF, Optimized kernel: 0.761**2 * RBF(length_scale=0.146) + WhiteKernel(noise_level=0.001)
Next points and acquisition values:
EI : [0.16105448 0.57812408 1.04298657] Value: 0.020452948773569242
PI : [0.16125928 0.50960919 1.09067087] Value: 0.4911797406784112
LCB: [0.49254904 0.6116888  0.33863169] Value: -0.06489443984402606


C:\Users\nmnko\anaconda3\Lib\site-packages\sklearn\gaussian_process\kernels.py:445: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 0.001. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(



Kernel: Matern, Optimized kernel: 0.73**2 * Matern(length_scale=0.214, nu=2.5) + WhiteKernel(noise_level=0.001)
Next points and acquisition values:
EI : [0.15889153 0.45038836 1.14641092] Value: 0.015140163387114351
PI : [0.1584473  0.50922386 1.09170296] Value: 0.48363201595595506
LCB: [0.491917   0.61157875 0.3391345 ] Value: -0.06509055390148671


C:\Users\nmnko\anaconda3\Lib\site-packages\sklearn\gaussian_process\kernels.py:445: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 0.001. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(



Kernel: RationalQuadratic, Optimized kernel: 0.765**2 * RationalQuadratic(alpha=0.225, length_scale=0.178) + WhiteKernel(noise_level=0.001)
Next points and acquisition values:
EI : [0.16224258 0.47551067 1.11160813] Value: 0.00813640619881498
PI : [0.15998741 0.50469482 1.08739082] Value: 0.4662413601014579
LCB: [0.49222467 0.6115505  0.33999858] Value: -0.06529410873529044
